# Register Neo4j Aura Agent MCP Server

Registers a Neo4j Aura Agent as an external MCP server in Databricks using
Dynamic Client Registration (DCR). This handles OAuth discovery, client
registration, and Unity Catalog connection creation automatically.

## Prerequisites

1. A deployed Neo4j Aura Agent with External access and MCP server enabled
2. Run `setup-secrets.sh` to store `MCP_SERVER_URL` in Databricks secrets
3. `CREATE CONNECTION` privilege on the Unity Catalog metastore

## Step 1: Install dependencies

In [ ]:
%pip install -U databricks-mcp nest-asyncio
%restart_python

## Step 2: Configuration

Load the MCP server URL from Databricks secrets.

**Option A** uses secrets created by `setup-secrets.sh`.
**Option B** sets the URL directly (for quick testing).

In [ ]:
# Connection name for Unity Catalog
CONNECTION_NAME = "neo4j_aura_agent"

# Secret scope (must match setup-secrets.sh)
SECRET_SCOPE = "aura-mcp-secrets"

In [ ]:
# --- Option A: Load from Databricks secrets (recommended) ---
MCP_SERVER_URL = dbutils.secrets.get(SECRET_SCOPE, "mcp-server-url")

# --- Option B: Set directly (uncomment and fill in) ---
# MCP_SERVER_URL = "https://mcp.neo4j.io/agent?project_id=YOUR_PROJECT_ID&agent_id=YOUR_AGENT_ID"

print(f"MCP Server URL: {MCP_SERVER_URL[:60]}...")

## Step 3: Register via Dynamic Client Registration

This cell:
1. Probes the MCP server with a POST request to discover OAuth endpoints (RFC 9728 → RFC 8414)
2. Registers an OAuth client via dynamic registration (RFC 7591)
3. Creates a Unity Catalog HTTP connection with MCP enabled

**Note:** The `databricks_mcp` library's `register_mcp_server_via_dcr` sends a GET
for the initial probe, but Neo4j's MCP server only accepts POST (returning 405).
We do the probe manually and then hand off to the library's individual functions.

In [ ]:
import re
import requests
from databricks.sdk import WorkspaceClient
from databricks_mcp.connector import (
    discover_authorization_server_metadata,
    perform_dynamic_client_registration,
    create_uc_connection,
)

workspace_client = WorkspaceClient()

# ── Probe the MCP server with POST ──────────────────────────────
# The databricks_mcp library sends GET, which Neo4j rejects with 405.
# Neo4j's MCP server requires POST, so we do this step manually.
probe_payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {"name": "databricks-mcp-register", "version": "1.0.0"},
    },
}
resp = requests.post(MCP_SERVER_URL, json=probe_payload, timeout=30)
assert resp.status_code == 401, f"Expected 401 from MCP server, got {resp.status_code}"

www_auth = resp.headers.get("www-authenticate", "")
match = re.search(r'resource_metadata=(\S+)', www_auth)
assert match, "No resource_metadata URL found in WWW-Authenticate header"
resource_metadata_url = match.group(1).strip(",")

# Fetch the Protected Resource Metadata (RFC 9728)
resource_meta = requests.get(resource_metadata_url, timeout=30).json()
print(f"Authorization server: {resource_meta.get('authorization_servers', ['?'])[0]}")

# ── Hand off to databricks_mcp library ──────────────────────────
# discover_authorization_server_metadata: fetches RFC 8414 metadata
as_meta = discover_authorization_server_metadata(resource_meta)
print(f"Token endpoint: {as_meta.get('token_endpoint')}")

# perform_dynamic_client_registration: registers client with Databricks redirect URI
dcr_result = perform_dynamic_client_registration(as_meta, resource_meta, www_auth, workspace_client)
print(f"Client registered: {dcr_result.get('client_id', '')[:20]}...")

# create_uc_connection: creates the Unity Catalog HTTP connection
connection_url = create_uc_connection(MCP_SERVER_URL, CONNECTION_NAME, dcr_result, workspace_client)
print(f"\nConnection created: {connection_url}")

## Step 4: Log in to the connection

The connection uses per-user OAuth. Each user must authenticate with Neo4j
through Databricks before tools are accessible. Visit the connection page
and click **Log in**, or use the MCP server in AI Playground (which triggers
the login automatically).

In [ ]:
# Print the connection URL for login
print(f"Log in to the connection before proceeding:")
print(f"  {connection_url}")
print(f"\nOr use AI Playground: Tools > MCP Servers > External MCP servers > {CONNECTION_NAME}")

## Step 5: Verify the connection (after login)

After logging in through the connection page or AI Playground, run this cell
to list the agent's tools.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from databricks_mcp import DatabricksMCPClient

host = workspace_client.config.host
proxy_url = f"{host}/api/2.0/mcp/external/{CONNECTION_NAME}"

mcp_client = DatabricksMCPClient(
    server_url=proxy_url,
    workspace_client=workspace_client,
)

tools = mcp_client.list_tools()
print(f"Available tools: {len(tools)}")
for tool in tools:
    print(f"  - {tool.name}: {tool.description[:80]}..." if len(tool.description) > 80 else f"  - {tool.name}: {tool.description}")

## Next steps

The MCP server is now available in:
- **AI Playground**: Select it under Tools > MCP Servers > External MCP servers
- **Agents**: Use the proxy URL in your agent code:
  ```python
  MANAGED_MCP_SERVER_URLS = [
      f"{host}/api/2.0/mcp/external/neo4j_aura_agent"
  ]
  ```
- **Genie Code**: Add the MCP server from the MCP server settings

To share access, grant `USE CONNECTION` on the Unity Catalog connection.